# 📝 과제 01: 온라인 쇼핑 상담 모델 DPO 정렬 학습

## 만들 결과

이 과제는 **교안 01(DPO 선호 정렬: 노무 상담 모델이 범위 밖 질문을 거절하도록 학습하기)** 의 과제입니다. 교안 01의 DPO 정렬을 SFT 단원의 과제에서 만든 온라인 쇼핑 상담 SFT 모델에 같은 순서로 적용해, 범위 밖 질문에 지어 답하던 모델이 거절하도록 beta 0.1로 학습합니다. 에폭마다 저장한 어댑터는 과제 02에서 평가해 정렬 모델을 고릅니다.

- 질문 다섯 개의 정렬 전후 답을 나란히 둔 `check_table`과 범위 밖 질문의 거절 수 `sft_refusal_count`, `dpo_refusal_count`
- 범위 밖 쌍과 쇼핑 질문 쌍을 합친 학습 쌍 `train_pairs`, 검증 쌍 `val_pairs`
- 에폭마다 저장한 어댑터 `output/commerce_dpo_beta0.1/checkpoint-<갱신 횟수>` 3개와 검증 보상 지표 테이블 `reward_table`

## 준비: 데이터와 제공 코드

- **SFT 모델:** SFT 단원의 과제 02에서 고른 쇼핑 상담 SFT 모델을 허깅페이스 허브(`muahseol/qwen3.5-0.8b-commerce-sft`)에서 내려받습니다(처음 한 번 약 3GB). 교안과 같은 RunPod GPU에서 실행하고, 판사를 쓰지 않아 API 키는 필요 없습니다.
- **자료:** `data/과제/commerce_train.jsonl`, `commerce_val.jsonl`(쇼핑 질문 쌍에 씁니다), `commerce_out_of_scope.jsonl`(범위 밖 질문 40개. `split`이 `train`인 32개는 학습 쌍, `val`인 8개는 검증 쌍과 정렬 전후 확인 질문에 씁니다), 일반 질문 `data/본실습/general_questions.jsonl`(정답 답변 키 `reference_answer`).
- **제공 코드:** 아래 셀을 차례로 실행합니다. SFT 모델 `sft_model`과 `tokenizer`, `SYSTEM_PROMPT_COMMERCE`, 거절 문장 `REFUSAL_ANSWER_COMMERCE`, 함수 `generate_replies(model, questions, system_prompt)`(그리디 생성), `load_jsonl(path)`, `sample_replies(model, questions, system_prompt, num_samples=4)`, `to_preference(question, chosen, rejected, system_prompt)`, MLflow 준비(실험 `commerce-dpo`), 데이터 `train_rows`, `val_rows`, `out_of_scope_train_rows`, `out_of_scope_val_rows`, 정렬 전후 확인 질문 `check_questions`, `check_kinds`, `reference_answers`와 함수 `answer_check_questions(model)`가 생깁니다.
- **교안 01과 다른 점:** beta 0.1 한 번만 학습하고, `max_length`는 512입니다(질문과 256토큰 답이 모두 들어가는 길이라 교안(1024)보다 작게 잡아 메모리를 아낍니다).
- **작성:** 코드 셀의 `# TODO (번호)`가 요구사항 번호입니다. `...`를 채우고, 문항 끝의 `[자가채점]` 셀을 실행합니다. 뒤 문항이 앞 문항의 변수를 쓰므로 1번부터 차례로 풉니다.

In [ ]:
import json
import platform
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

# 테이블의 긴 답변이 잘리지 않게 칸의 글자 수 제한을 풉니다
pd.set_option("display.max_colwidth", None)

# 그래프 글꼴: 영문, 숫자, 기호는 DejaVu Sans, 한글은 운영체제의 한글 글꼴이 맡습니다
from matplotlib import font_manager

if platform.system() == "Windows":
    korean_font = "Malgun Gothic"
elif platform.system() == "Darwin":
    korean_font = "AppleGothic"
else:
    korean_font = "NanumGothic"
    # 리눅스(RunPod): 글꼴을 설치하기 전에 만든 matplotlib 글꼴 목록에는 나눔고딕이 없으므로, 설치된 파일을 직접 등록합니다
    for font_path in font_manager.findSystemFonts():
        if "NanumGothic" in font_path:
            font_manager.fontManager.addfont(font_path)
plt.rcParams["font.family"] = ["DejaVu Sans", korean_font]
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호가 네모로 깨지지 않게 합니다

In [ ]:
# SFT 단원의 과제 02에서 고른 온라인 쇼핑 상담 SFT 모델을 허깅페이스 허브에 올려 둔 저장소. 처음 실행할 때 약 3GB를 내려받습니다
# SFT 단원에서 직접 저장한 모델을 쓰려면 이 값을 그 폴더 경로(SFT 단원 폴더의 output/commerce_sft_final)로 바꿉니다
SFT_MODEL_NAME = "muahseol/qwen3.5-0.8b-commerce-sft"

tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL_NAME)
sft_model = AutoModelForCausalLM.from_pretrained(SFT_MODEL_NAME, dtype=torch.float32).to(device)
print("SFT 모델:", SFT_MODEL_NAME, "| 파라미터 수:", f"{sft_model.num_parameters():,}")

In [ ]:
# SFT 단원의 과제 데이터의 시스템 프롬프트와 범위 밖 거절 답입니다(학습 데이터의 messages와 글자까지 같습니다)
SYSTEM_PROMPT_COMMERCE = "너는 온라인 쇼핑 상담 챗봇이다. 전자상거래법 조문에 근거해 존댓말로 간결히 답한다."
REFUSAL_ANSWER_COMMERCE = "이 상담 챗봇이 다루는 전자상거래법 조문으로는 답할 수 없는 질문입니다. 이 내용은 담당 기관이나 전문가에게 문의해 주세요."

# 대화 끝 토큰 <|im_end|>의 id. 생성이 이 토큰에서 멈추도록 eos_token_id로 넘깁니다
END_OF_TURN_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")


def generate_replies(model, questions, system_prompt, max_new_tokens=256):
    """질문을 32개씩 묶어 생성하고, 질문 순서대로 답 목록을 돌려줍니다. system_prompt=None 이면 시스템 메시지 없이 묻습니다."""
    # 여러 질문을 묶으면 짧은 질문을 패딩으로 채워 길이를 맞춥니다. 답은 오른쪽 끝에 이어 쓰므로 패딩은 왼쪽에 둡니다
    tokenizer.padding_side = "left"
    answers = []
    for start in range(0, len(questions), 32):
        # generate_reply 와 같은 chat template으로 질문을 감쌉니다
        prompts = []
        for question in questions[start:start + 32]:
            messages = []
            if system_prompt is not None:
                messages.append({"role": "system", "content": system_prompt})
            messages.append({"role": "user", "content": question})
            prompts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False))
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)

        # generate_reply 와 같은 그리디 생성
        with torch.no_grad():
            output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                        eos_token_id=END_OF_TURN_ID, pad_token_id=tokenizer.pad_token_id)

        # 입력 길이 뒤의 새 토큰만 글자로 바꿉니다(먼저 끝난 답 뒤의 패딩은 skip_special_tokens 로 빠집니다)
        for new_ids in output_ids[:, inputs["input_ids"].shape[1]:]:
            answers.append(tokenizer.decode(new_ids, skip_special_tokens=True))

    tokenizer.padding_side = "right"  # 학습기가 쓰는 오른쪽 패딩으로 되돌립니다
    return answers


def load_jsonl(path):
    """JSONL 파일을 한 줄에 딕셔너리 하나씩 읽어 리스트로 돌려줍니다(SFT 단원과 같은 함수)."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

In [ ]:
# 교안 01과 같은 함수: 질문마다 샘플링한 답 여러 개, 대화형 선호 쌍 한 행
def sample_replies(model, questions, system_prompt, num_samples=4, max_new_tokens=256):
    """질문마다 샘플링으로 답을 num_samples 개 만들어, 질문 순서대로 답 리스트를 담은 리스트를 돌려줍니다."""
    all_samples = []
    for question in questions:
        # generate_replies 와 같은 chat template으로 질문 하나를 감쌉니다
        messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": question}]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        # 확률에 따라 토큰을 뽑아(샘플링) 같은 질문에서 서로 다른 답을 num_samples 개 만듭니다
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,                        # 그리디 대신 확률대로 뽑습니다
                temperature=1.0,                       # 확률을 키우거나 줄이지 않습니다(작을수록 그리디에 가까움)
                top_p=0.95,                            # 확률 합 95% 안의 토큰에서만 뽑아 아주 드문 토큰을 뺍니다
                num_return_sequences=num_samples,      # 한 질문에서 답을 이 개수만큼 만듭니다
                eos_token_id=END_OF_TURN_ID,
                pad_token_id=tokenizer.pad_token_id,
            )

        # 답마다 입력 길이 뒤의 새 토큰만 글자로 바꿉니다
        samples = []
        for new_ids in output_ids[:, inputs["input_ids"].shape[1]:]:
            samples.append(tokenizer.decode(new_ids, skip_special_tokens=True))
        all_samples.append(samples)
    return all_samples


def to_preference(question, chosen, rejected, system_prompt):
    """질문 하나와 두 답을 DPOTrainer 가 받는 대화형 선호 쌍 한 행으로 바꿉니다."""
    return {
        "prompt": [{"role": "system", "content": system_prompt}, {"role": "user", "content": question}],
        "chosen": [{"role": "assistant", "content": chosen}],      # 더 나은 답
        "rejected": [{"role": "assistant", "content": rejected}],  # 더 나쁜 답
        "chat_template_kwargs": {"enable_thinking": False},         # 생성할 때와 같은 빈 생각 블록을 붙입니다
    }

In [ ]:
import mlflow

# 실험 기록은 output/mlflow.db 한 파일에 남깁니다(별도 서버 없이 sqlite)
mlflow.set_tracking_uri(f"sqlite:///{output_dir / 'mlflow.db'}")
mlflow.set_experiment("commerce-dpo")
print("MLflow 기록 위치:", mlflow.get_tracking_uri())

In [ ]:
# SFT 단원의 과제와 같은 학습, 검증 데이터(쇼핑 질문 쌍에 씁니다)
train_rows = load_jsonl(data_dir / "과제" / "commerce_train.jsonl")
val_rows = load_jsonl(data_dir / "과제" / "commerce_val.jsonl")

# 범위 밖 질문: split 이 train 이면 학습 쌍, val 이면 검증 쌍에 씁니다
out_of_scope_rows = load_jsonl(data_dir / "과제" / "commerce_out_of_scope.jsonl")
out_of_scope_train_rows = []
out_of_scope_val_rows = []
for row in out_of_scope_rows:
    if row["split"] == "train":
        out_of_scope_train_rows.append(row)
    else:
        out_of_scope_val_rows.append(row)
print("학습 행:", len(train_rows), "| 검증 행:", len(val_rows), "| 범위 밖 학습 질문:", len(out_of_scope_train_rows), "| 범위 밖 검증 질문:", len(out_of_scope_val_rows))

### 정렬 전후에 물을 질문

교안 01 3-4와 같은 구성의 질문 다섯 개입니다. 실행만 합니다.

| 종류 | 개수 | 보는 것 |
|---|---|---|
| 범위 밖 질문(검증) | 2 | DPO 학습에 쓰지 않은 범위 밖 질문을 거절하는지 |
| 쇼핑 질문(검증) | 1 | 학습에 쓰지 않은 쇼핑 질문에 거절하지 않고 답하는지(과잉 거절) |
| 쇼핑 질문(학습) | 1 | SFT로 배운 답을 지키는지 |
| 일반 질문 | 1 | 시스템 프롬프트 없이 상식 답을 지키는지 |

In [ ]:
# 쇼핑과 상관없는 상식 질문(정답 답변은 reference_answer)
general_rows = load_jsonl(data_dir / "본실습" / "general_questions.jsonl")

# 정렬 전후에 똑같이 물을 질문 다섯 개와 그 종류. 범위 밖 질문은 검증 질문 앞 두 개입니다
check_questions = [out_of_scope_val_rows[0]["question"], out_of_scope_val_rows[1]["question"], val_rows[0]["question"],
                   train_rows[90]["question"], general_rows[12]["question"]]
check_kinds = ["범위 밖 질문(검증)", "범위 밖 질문(검증)", "쇼핑 질문(검증)", "쇼핑 질문(학습)", "일반 질문"]

# 질문마다 정답 답변: 범위 밖 질문은 거절 문장, 쇼핑 질문은 answer, 일반 질문은 reference_answer
reference_answers = [REFUSAL_ANSWER_COMMERCE, REFUSAL_ANSWER_COMMERCE, val_rows[0]["answer"],
                     train_rows[90]["answer"], general_rows[12]["reference_answer"]]


def answer_check_questions(model):
    """질문 다섯 개에 model이 그리디 생성으로 한 답을 같은 순서의 리스트로 돌려줍니다."""
    # 앞 네 질문은 서비스와 같은 쇼핑 상담 시스템 프롬프트로 묻습니다
    answers = generate_replies(model, check_questions[:4], SYSTEM_PROMPT_COMMERCE)

    # 일반 질문은 시스템 프롬프트 없이 묻습니다(쇼핑 프롬프트로 물으면 거절이 상식을 잃은 것인지 가를 수 없습니다)
    answers += generate_replies(model, check_questions[4:], None)
    return answers

## 1. 정렬 전 답과 답 여러 개 뽑기: answer_check_questions, sample_replies

**목적:** DPO가 고칠 실패(범위 밖 질문에 지어 답하기)를 정렬 전에 남겨 두고, 더 나쁜 답(`rejected`) 후보를 정렬할 SFT 모델이 실제로 내는 답에서 모읍니다(on-policy).

**요구사항**
- (1) 열이 `"종류"`, `"질문"`, `"정답 답변"`, `"SFT 모델 답변"` 순서인 `pd.DataFrame` **`check_table`** 을 만듭니다. `"종류"`는 `check_kinds`, `"질문"`은 `check_questions`, `"정답 답변"`은 `reference_answers`, `"SFT 모델 답변"`은 `answer_check_questions(sft_model)`이 돌려준 답입니다(답마다 비어 있지 않고 질문 글과 다른 문자열이며, 다섯 답 전체가 `reference_answers`와 같지 않아야 합니다).
- (2) `"종류"`가 `"범위 밖 질문(검증)"`인 두 행의 `"SFT 모델 답변"` 가운데 거절 키워드 `"답할 수 없"`가 든 답의 수를 정수(`int`) **`sft_refusal_count`** 에 담습니다.
- (3) `out_of_scope_train_rows`의 `question`을 순서대로 담은 리스트 **`train_questions`** 를 만듭니다(32개).
- (4) `out_of_scope_val_rows`의 `question`을 순서대로 담은 리스트 **`val_questions`** 를 만듭니다(8개).
- (5) 셀의 `torch.manual_seed(0)` 다음에 `sample_replies`로 `train_questions`의 답을 뽑아 리스트 **`train_samples`** 에 담습니다(`sft_model`, `SYSTEM_PROMPT_COMMERCE`, `num_samples`는 기본값 4). 질문 32개마다 답 4개짜리 리스트가 하나씩 있습니다.
- (6) 같은 방법으로 `val_questions`의 답을 뽑아 리스트 **`val_samples`** 에 담습니다(질문 8개마다 답 4개).

<details><summary>힌트</summary>

```text
접근방법: 교안 01 3-4 정렬 전 답, 4-2 SFT 모델의 답 여러 개 뽑기 셀과 같은 방법입니다.
세부구현:
1. 준비의 세 리스트와 SFT 모델에 물은 답 리스트를 열 이름을 키로 둔 딕셔너리에 담아 테이블로 만듭니다.
2. "종류" 열과 "SFT 모델 답변" 열을 zip 으로 함께 돌며, 범위 밖 행이면서 거절 키워드가 든 답만 셉니다.
3. 범위 밖 학습, 검증 행에서 question 만 모읍니다.
4. 시드 줄 다음에 sample_replies 를 학습 질문, 검증 질문 순서로 부릅니다.
```

</details>

In [ ]:
# TODO (1): 열이 "종류", "질문", "정답 답변", "SFT 모델 답변" 인 check_table 을 만드세요(답은 answer_check_questions 로)
check_table = ...
display(check_table)

In [ ]:
# TODO (2): 범위 밖 질문 두 행의 "SFT 모델 답변" 가운데 거절 키워드가 든 답의 수를 sft_refusal_count 에 담으세요
sft_refusal_count = ...
print("범위 밖 질문 2개 가운데 SFT 모델이 거절한 답:", sft_refusal_count)

In [ ]:
# TODO (3): out_of_scope_train_rows 의 question 을 순서대로 train_questions 에 담으세요
train_questions = ...

# TODO (4): out_of_scope_val_rows 의 question 을 순서대로 val_questions 에 담으세요
val_questions = ...

# TODO (5): 시드 줄은 그대로 두고, sample_replies 로 학습 질문의 답을 뽑아 train_samples 에 담으세요
torch.manual_seed(0)
train_samples = ...

# TODO (6): 같은 방법으로 val_questions 의 답을 뽑아 val_samples 에 담으세요
val_samples = ...
print("학습 질문:", len(train_samples), "| 검증 질문:", len(val_samples))

In [ ]:
# [자가채점] 정렬 전 답 테이블과 거절 수, 질문 리스트와 답 묶음의 모양을 확인합니다(샘플 답 내용은 실행마다 달라 보지 않습니다)
# check_table 의 열과 준비의 세 리스트
assert isinstance(check_table, pd.DataFrame), "check_table은 pd.DataFrame으로 만든 테이블입니다. `...`를 채웠는지 확인하세요."
assert list(check_table.columns) == ["종류", "질문", "정답 답변", "SFT 모델 답변"], f"check_table의 열이 {list(check_table.columns)}입니다. 열은 \"종류\", \"질문\", \"정답 답변\", \"SFT 모델 답변\" 순서입니다."
assert list(check_table["종류"]) == check_kinds and list(check_table["질문"]) == check_questions and list(check_table["정답 답변"]) == reference_answers, "\"종류\", \"질문\", \"정답 답변\" 열은 check_kinds, check_questions, reference_answers를 순서 그대로 담습니다."

# SFT 모델 답변: 비어 있지 않고 질문 글을 그대로 옮긴 것이 아니며, 다섯 답 전체가 정답 답변을 옮긴 것이 아닌지
for question, answer in zip(check_questions, check_table["SFT 모델 답변"]):
    assert isinstance(answer, str) and answer.strip() != "", "\"SFT 모델 답변\"이 비어 있습니다. answer_check_questions(sft_model)의 반환값을 담으세요."
    assert answer != question, "\"SFT 모델 답변\"에 질문 글이 들어 있습니다. answer_check_questions(sft_model)의 반환값을 담으세요."
assert list(check_table["SFT 모델 답변"]) != reference_answers, "\"SFT 모델 답변\"이 \"정답 답변\"과 다섯 개 모두 같습니다. reference_answers가 아니라 answer_check_questions(sft_model)의 반환값을 담으세요."

# 범위 밖 두 행의 거절 수: 두 행만 세므로 0부터 2 사이의 정수
assert isinstance(sft_refusal_count, int), f"sft_refusal_count가 {sft_refusal_count!r}입니다. `...`를 채우고 거절한 답의 수를 정수로 담으세요(`.sum()`으로 셌다면 `int()`로 바꾸세요)."
assert 0 <= sft_refusal_count <= 2, f"sft_refusal_count가 {sft_refusal_count}입니다. \"종류\"가 \"범위 밖 질문(검증)\"인 두 행의 \"SFT 모델 답변\"만 세므로 0부터 2 사이입니다."

# 범위 밖 학습, 검증 질문: 개수, 글자, 행 순서의 첫 질문과 마지막 질문
for questions, rows, count, name in [(train_questions, out_of_scope_train_rows, 32, "train_questions"), (val_questions, out_of_scope_val_rows, 8, "val_questions")]:
    assert isinstance(questions, list) and len(questions) == count, f"{name}는 질문 {count}개를 담은 리스트입니다. `...`를 채웠는지 확인하세요."
    for question in questions:
        assert isinstance(question, str), f"{name}에 글자가 아닌 값이 있습니다. 행 전체가 아니라 행의 question만 담으세요."
    assert questions[0] == rows[0]["question"] and questions[-1] == rows[-1]["question"], f"{name}의 첫 질문이나 마지막 질문이 행 순서와 다릅니다. 행 순서대로 담으세요."

# 답 묶음: 질문마다 답 4개짜리 리스트
assert isinstance(train_samples, list), "train_samples는 sample_replies가 돌려준 리스트입니다. `...`를 채웠는지 확인하세요."
assert len(train_samples) == 32, f"train_samples가 {len(train_samples)}개입니다. train_questions(32개)를 sample_replies에 넘기세요."
assert isinstance(val_samples, list), "val_samples는 sample_replies가 돌려준 리스트입니다. `...`를 채웠는지 확인하세요."
assert len(val_samples) == 8, f"val_samples가 {len(val_samples)}개입니다. val_questions(8개)를 sample_replies에 넘기세요."
for samples in train_samples + val_samples:
    assert isinstance(samples, list) and len(samples) == 4, "질문마다 답 4개짜리 리스트가 있어야 합니다. sample_replies의 반환값을 그대로 담고 num_samples는 기본값을 쓰세요."
print("✅ 통과!")

## 2. 선호 쌍 만들기: make_out_of_scope_pairs, to_preference, Dataset.from_list

**목적:** SFT 모델이 실제로 지어 낸 답을 `rejected`, 고정 거절 문장을 `chosen`으로 둔 범위 밖 쌍으로 거절을 가르치고, 쇼핑 질문까지 거절하지 않도록(과잉 거절) 쇼핑 질문 쌍을 함께 만들어 `DPOTrainer`가 받는 대화형 선호 쌍으로 바꿉니다. 쇼핑 질문 쌍은 범위 밖 쌍의 4분의 3입니다. 교안 01처럼 절반만 넣으면 이 SFT 모델은 쇼핑 질문까지 거절해(교안 01 4-4의 과잉 거절) 쌍을 늘렸습니다.

**요구사항**
- (1) 질문 행 리스트 `rows`와 같은 순서의 답 묶음 `samples`(1번 `train_samples`와 같은 모양)를 받아 범위 밖 쌍 리스트를 질문 순서대로 돌려주는 함수 **`make_out_of_scope_pairs(rows, samples)`** 를 만듭니다.
- (2) 질문마다 답을 앞에서부터 보며 `"답할 수 없"`가 없고 빈 답도 아닌(`strip()` 결과가 `""`가 아닌) 첫 답을 `rejected`로 고릅니다.
- (3) 고른 답이 있으면 키가 `"question"`(행의 `question`), `"chosen"`(`REFUSAL_ANSWER_COMMERCE`), `"rejected"` 순서인 딕셔너리를 리스트에 더하고, 없으면 그 질문을 건너뜁니다.
- (4) 이 함수로 학습 질문(`out_of_scope_train_rows`, `train_samples`)의 쌍 **`train_out_of_scope_pairs`** 와 검증 질문(`out_of_scope_val_rows`, `val_samples`)의 쌍 **`val_out_of_scope_pairs`** 를 만듭니다.
- (5) 학습 질문과 검증 질문에서 건너뛴 질문 수를 각각 정수 **`train_skipped_count`**, **`val_skipped_count`** 에 담습니다(쌍 수와 더하면 질문 수 32, 8).
- (6) 교안 01처럼 거절 답 행을 먼저 거릅니다. `train_rows`에서 `answer`에 `"답할 수 없"`가 없는 행만 순서대로 담은 리스트 **`answer_rows`**(128행)와, `val_rows`에서 같은 방법으로 거른 리스트 **`val_answer_rows`**(34행)를 만듭니다.
- (7) `answer_rows[1::3][:24]`의 행마다 키가 `"question"`(행의 `question`), `"chosen"`(행의 `answer`), `"rejected"`(`REFUSAL_ANSWER_COMMERCE`) 순서인 딕셔너리를 담은 리스트 **`train_shopping_pairs`** 를 만듭니다(24쌍).
- (8) 거절 답 행을 거른 검증 행에서 8행 간격으로 4행, 곧 `val_answer_rows[::8][:4]`의 행으로 같은 모양의 리스트 **`val_shopping_pairs`** 를 만듭니다(4쌍).
- (9) `train_out_of_scope_pairs + train_shopping_pairs` 순서로 쌍마다 `to_preference(question, chosen, rejected, SYSTEM_PROMPT_COMMERCE)`가 돌려준 행을 모아, `Dataset.from_list`로 학습 쌍 **`train_pairs`** 를 만듭니다(열은 `prompt`, `chosen`, `rejected`, `chat_template_kwargs`. 쌍 하나가 한 행).
- (10) `val_out_of_scope_pairs + val_shopping_pairs`로 같은 방법의 검증 쌍 **`val_pairs`** 를 만듭니다.

<details><summary>힌트</summary>

```text
접근방법: 교안 01 4-3 범위 밖 쌍 만들기, 4-4 노무 질문 쌍 만들기, 4-5 대화형 Dataset으로 바꾸기 셀과 같은 방법입니다.
교안은 범위 밖 쌍 반복문을 셀에 펼쳐 쓰지만, 여기서는 학습, 검증 질문에 두 번 쓰므로 함수로 둡니다.
세부구현:
1. 함수 안에서 질문 행과 답 묶음을 zip 으로 함께 돌고, 답 묶음 안에서 조건에 맞는 첫 답을 찾으면 break 로 멈춥니다.
2. 못 찾은 경우를 가르려면 답 묶음을 돌기 전에 None 을 넣어 둡니다.
3. 건너뛴 질문 수는 질문 행 수에서 만든 쌍 수를 뺀 값입니다.
4. 쇼핑 질문 쌍은 범위 밖 쌍과 chosen, rejected 자리가 반대입니다.
5. 두 쌍 리스트를 + 로 이어 돌며 to_preference 의 반환값을 모으고, 다 모은 리스트를 Dataset 으로 바꿉니다.
```

</details>

In [ ]:
# TODO (1): 범위 밖 쌍 리스트를 질문 순서대로 돌려주도록 아래 함수를 채우세요
def make_out_of_scope_pairs(rows, samples):
    """질문마다 거절도 빈 답도 아닌 첫 샘플 답을 rejected, 고정 거절 문장을 chosen 으로 둔 쌍 리스트를 돌려줍니다."""
    pairs = []
    # TODO (2): 질문마다 거절도 빈 답도 아닌 첫 답을 rejected 로 고르세요
    # TODO (3): 고른 답이 있으면 키가 question, chosen, rejected 인 딕셔너리를 pairs 에 더하세요
    ...
    return pairs


# TODO (4): 학습 질문과 검증 질문으로 범위 밖 쌍을 만드세요
train_out_of_scope_pairs = ...
val_out_of_scope_pairs = ...

# TODO (5): 학습 질문과 검증 질문에서 건너뛴 질문 수를 담으세요
train_skipped_count = ...
val_skipped_count = ...
print("학습 범위 밖 쌍:", len(train_out_of_scope_pairs), "| 건너뛴 학습 질문:", train_skipped_count)
print("검증 범위 밖 쌍:", len(val_out_of_scope_pairs), "| 건너뛴 검증 질문:", val_skipped_count)

In [ ]:
# TODO (6): train_rows, val_rows 에서 answer 에 거절 키워드가 없는 행만 순서대로 answer_rows, val_answer_rows 에 담으세요
answer_rows = ...
val_answer_rows = ...

# TODO (7): answer_rows[1::3][:24] 의 행으로 학습 쇼핑 질문 쌍 train_shopping_pairs 를 만드세요
train_shopping_pairs = ...

# TODO (8): val_answer_rows[::8][:4] 의 행으로 검증 쇼핑 질문 쌍 val_shopping_pairs 를 만드세요
val_shopping_pairs = ...
print("거절 아닌 학습 행:", len(answer_rows), "| 거절 아닌 검증 행:", len(val_answer_rows), "| 학습 쇼핑 질문 쌍:", len(train_shopping_pairs), "| 검증 쇼핑 질문 쌍:", len(val_shopping_pairs))

In [ ]:
from datasets import Dataset

# TODO (9): 범위 밖 쌍 뒤에 쇼핑 질문 쌍을 이어 to_preference 로 바꾸고, Dataset 학습 쌍 train_pairs 를 만드세요
train_pairs = ...

# TODO (10): 같은 방법으로 검증 쌍 val_pairs 를 만드세요
val_pairs = ...
print("학습 쌍:", len(train_pairs), "| 검증 쌍:", len(val_pairs))

In [ ]:
# [자가채점] 범위 밖 쌍 함수와 쌍, 쇼핑 질문 쌍, 대화형 Dataset 을 확인합니다
# 작은 예로 함수 규칙 확인
test_refusal = "그 질문은 이 챗봇이 답할 수 없는 내용입니다."
test_rows = [{"question": "질문 1"}, {"question": "질문 2"}, {"question": "질문 3"}]
test_samples = [
    [test_refusal, "  ", "지어 낸 답 1", "지어 낸 답 2"],
    [test_refusal, "", test_refusal, " "],
    ["지어 낸 답 3", test_refusal, test_refusal, test_refusal],
]
test_pairs = make_out_of_scope_pairs(test_rows, test_samples)
expected_test_pairs = [
    {"question": "질문 1", "chosen": REFUSAL_ANSWER_COMMERCE, "rejected": "지어 낸 답 1"},
    {"question": "질문 3", "chosen": REFUSAL_ANSWER_COMMERCE, "rejected": "지어 낸 답 3"},
]
assert isinstance(test_pairs, list) and test_pairs == expected_test_pairs, f"작은 예의 결과가 {test_pairs}입니다. 질문 1은 빈 답을 건너뛴 첫 지어 낸 답, 질문 2는 거절과 빈 답뿐이라 쌍 없음, 질문 3은 첫 답이 rejected 여야 합니다."
for pair in test_pairs:
    assert list(pair) == ["question", "chosen", "rejected"], f"쌍의 키가 {list(pair)}입니다. \"question\", \"chosen\", \"rejected\" 순서로 만드세요."

# 범위 밖 쌍: 그 묶음의 질문이 행 순서대로 한 번씩, chosen 은 거절 문장, rejected 는 그 질문의 샘플 답
for rows, samples, pairs, name in [(out_of_scope_train_rows, train_samples, train_out_of_scope_pairs, "train_out_of_scope_pairs"), (out_of_scope_val_rows, val_samples, val_out_of_scope_pairs, "val_out_of_scope_pairs")]:
    assert isinstance(pairs, list), f"{name}는 make_out_of_scope_pairs가 돌려준 쌍 리스트입니다. `...`를 채웠는지 확인하세요."
    samples_by_question = {}
    position_by_question = {}
    for position, row in enumerate(rows):
        samples_by_question[row["question"]] = samples[position]
        position_by_question[row["question"]] = position
    previous_position = -1
    for pair in pairs:
        assert pair["question"] in samples_by_question, f"{name}에 이 묶음의 범위 밖 질문이 아닌 질문이 있습니다. 학습 쌍은 out_of_scope_train_rows와 train_samples, 검증 쌍은 out_of_scope_val_rows와 val_samples로 만드세요."
        assert position_by_question[pair["question"]] > previous_position, f"{name}의 질문 순서가 행 순서와 다르거나 같은 질문이 두 번 있습니다. 질문 순서대로 한 번씩 만드세요."
        previous_position = position_by_question[pair["question"]]
        assert pair["chosen"] == REFUSAL_ANSWER_COMMERCE, "범위 밖 쌍의 chosen은 REFUSAL_ANSWER_COMMERCE입니다."
        assert "답할 수 없" not in pair["rejected"] and pair["rejected"].strip() != "", "rejected에 거절 키워드가 있거나 빈 답입니다. 거절도 빈 답도 아닌 답을 고르세요."
        assert pair["rejected"] in samples_by_question[pair["question"]], "rejected는 그 질문의 샘플 답 가운데 하나여야 합니다. 질문과 답 묶음을 같은 순서로 짝지었는지 확인하세요."
assert isinstance(train_skipped_count, int) and isinstance(val_skipped_count, int), f"건너뛴 수가 {train_skipped_count!r}, {val_skipped_count!r}입니다. `...`를 채우고 질문 수에서 쌍 수를 뺀 정수를 담으세요."
assert len(train_out_of_scope_pairs) + train_skipped_count == 32, f"학습 쌍 {len(train_out_of_scope_pairs)}개와 건너뛴 {train_skipped_count}개의 합이 학습 질문 32개와 다릅니다."
assert len(val_out_of_scope_pairs) + val_skipped_count == 8, f"검증 쌍 {len(val_out_of_scope_pairs)}개와 건너뛴 {val_skipped_count}개의 합이 검증 질문 8개와 다릅니다."

# 거절 답 행을 거른 학습, 검증 행: 행 수, 거절 키워드 없음, 원래 행 순서 그대로 한 번씩
for kept_rows, source_rows, count, name, source_name in [(answer_rows, train_rows, 128, "answer_rows", "train_rows"), (val_answer_rows, val_rows, 34, "val_answer_rows", "val_rows")]:
    assert isinstance(kept_rows, list) and len(kept_rows) == count, f"{name}는 {source_name}에서 answer에 \"답할 수 없\"가 없는 행 {count}개를 담은 리스트입니다. `...`를 채웠는지 확인하세요."
    previous_position = -1
    for row in kept_rows:
        assert row in source_rows, f"{name}에 {source_name}의 행이 아닌 값이 있습니다. 행 딕셔너리를 그대로 담으세요."
        assert "답할 수 없" not in row["answer"], f"{name}에 answer에 \"답할 수 없\"가 든 행이 있습니다. 거절 답 행은 빼세요."
        assert source_rows.index(row) > previous_position, f"{name}의 행 순서가 {source_name}와 다르거나 같은 행이 두 번 있습니다. {source_name} 순서대로 한 번씩 담으세요."
        previous_position = source_rows.index(row)

# 쇼핑 질문 쌍: 지문의 슬라이싱으로 고른 행, 키 순서, chosen 과 rejected 자리
for pairs, source_rows, count, name in [(train_shopping_pairs, answer_rows[1::3][:24], 24, "train_shopping_pairs"), (val_shopping_pairs, val_answer_rows[::8][:4], 4, "val_shopping_pairs")]:
    assert isinstance(pairs, list), f"{name}는 쌍 딕셔너리를 담은 리스트입니다. `...`를 채웠는지 확인하세요."
    assert len(pairs) == count, f"{name}가 {len(pairs)}쌍입니다. 지문의 슬라이싱으로 고른 {count}행마다 쌍 하나를 만드세요."
    for pair, row in zip(pairs, source_rows):
        assert list(pair) == ["question", "chosen", "rejected"], f"{name} 쌍의 키가 {list(pair)}입니다. \"question\", \"chosen\", \"rejected\" 순서로 만드세요."
        assert pair["question"] == row["question"], f"{name}의 질문 순서가 지문의 슬라이싱과 다릅니다. 고른 행을 순서대로 쓰세요."
        assert pair["chosen"] == row["answer"] and pair["rejected"] == REFUSAL_ANSWER_COMMERCE, f"{name}의 chosen은 행의 answer, rejected는 REFUSAL_ANSWER_COMMERCE입니다(범위 밖 쌍과 자리가 반대)."

# 대화형 Dataset: 열, 행 수, 첫 행과 마지막 행의 질문과 답
for dataset, pairs, name in [(train_pairs, train_out_of_scope_pairs + train_shopping_pairs, "train_pairs"), (val_pairs, val_out_of_scope_pairs + val_shopping_pairs, "val_pairs")]:
    assert isinstance(dataset, Dataset), f"{name}는 Dataset.from_list로 만든 Dataset입니다. `...`를 채웠는지 확인하세요."
    assert dataset.column_names == ["prompt", "chosen", "rejected", "chat_template_kwargs"], f"{name}의 열이 {dataset.column_names}입니다. 쌍마다 to_preference가 돌려준 행으로 만드세요."
    assert len(dataset) == len(pairs), f"{name}가 {len(dataset)}행입니다. 범위 밖 쌍과 쇼핑 질문 쌍을 이어 붙여 쌍마다 한 행을 만드세요."
    for index in [0, len(pairs) - 1]:
        row = dataset[index]
        assert [message["role"] for message in row["prompt"]] == ["system", "user"] and row["prompt"][0]["content"] == SYSTEM_PROMPT_COMMERCE, f"{name}의 prompt는 SYSTEM_PROMPT_COMMERCE 시스템 메시지와 질문입니다. to_preference에 SYSTEM_PROMPT_COMMERCE를 넘기세요."
        assert row["prompt"][1]["content"] == pairs[index]["question"], f"{name}의 행 순서가 다릅니다. 범위 밖 쌍 뒤에 쇼핑 질문 쌍을 이어 붙인 순서로 만드세요."
        assert row["chosen"][0]["content"] == pairs[index]["chosen"] and row["rejected"][0]["content"] == pairs[index]["rejected"], f"{name}의 chosen, rejected가 쌍과 다릅니다. to_preference의 인자 순서(질문, chosen, rejected, 시스템 프롬프트)를 확인하세요."
print("✅ 통과!")

## 3. LoRA 설정, DPO 학습 설정과 학습: LoraConfig, DPOConfig, DPOTrainer

**목적:** SFT 모델에 어댑터를 붙여 학습하고 어댑터를 끈 같은 모델을 참조 모델로 쓰는 학습기로 3에폭 학습해, 과제 02가 평가할 에폭별 어댑터를 저장하고 학습 기록을 MLflow run에 남깁니다.

**요구사항**
- (1) `r=8`, `lora_alpha=16`, `target_modules="all-linear"`, `lora_dropout=0.05`, `task_type="CAUSAL_LM"`인 `LoraConfig` **`lora_config`** 를 만듭니다.
- (2) `DPOConfig`에 아래 값만 넣어 **`dpo_config`** 를 만듭니다(`max_steps` 같은 다른 인자는 넣지 않습니다).

```text
output_dir                   str(output_dir / "commerce_dpo_beta0.1")
beta                         0.1
loss_type                    ["sigmoid", "sft"]
learning_rate                5e-5
num_train_epochs             3
per_device_train_batch_size  4
per_device_eval_batch_size   4
seed                         0
gradient_checkpointing       False
max_length                   512
eval_strategy                "epoch"
save_strategy                "epoch"
save_only_model              True
logging_steps                5
eval_on_start                True
report_to                    ["mlflow"]
run_name                     "commerce_dpo_beta_0.1"
```

- (3) `DPOTrainer`에 `model=sft_model`, `args=dpo_config`, `train_dataset=train_pairs`, `eval_dataset=val_pairs`, `processing_class=tokenizer`, `peft_config=lora_config`를 넘긴 **`trainer`** 를 만듭니다. `ref_model`은 넘기지 않습니다(`trainer.ref_model`이 `None`).
- (4) 셀의 MLflow run 안에서 `trainer.train()`으로 끝까지 학습합니다. 끝까지 학습하면 `trainer.state.global_step`이 올림(`len(train_pairs)` / 4) × 3입니다.

<details><summary>힌트</summary>

```text
접근방법: 교안 01 5-1 LoRA 설정과 참조 모델, 5-2 학습 설정, 5-3 학습 셀과 같은 방법입니다.
세부구현:
1. LoraConfig 는 교안과 같은 값입니다.
2. DPOConfig 는 지문 테이블의 값만 키워드 인자로 넣습니다. output_dir, max_length, run_name 은 교안과 다릅니다.
3. 학습기에는 참조 모델을 따로 넘기지 않고 LoRA 설정을 넘깁니다.
4. 학습은 셀의 with 블록 안에서 부릅니다.
```

</details>

In [ ]:
# 설정을 고쳐 다시 실행할 때는 준비의 SFT 모델 셀을 먼저 다시 실행합니다(학습기가 sft_model 에 어댑터를 붙입니다)
from peft import LoraConfig
from trl import DPOConfig, DPOTrainer

# TODO (1): 지문의 값으로 LoRA 설정 lora_config 를 만드세요
lora_config = ...

# TODO (2): 지문의 값 목록으로 DPO 학습 설정을 채우세요
dpo_config = DPOConfig(
    ...
)

# TODO (3): ref_model 없이 peft_config 를 넘겨 학습기 trainer 를 만드세요
trainer = ...

In [ ]:
# 다시 학습하려면 output/commerce_dpo_beta0.1 폴더를 지우고 준비의 SFT 모델 셀을 다시 실행한 뒤 3번 첫 셀부터 다시 실행합니다
with mlflow.start_run(run_name=dpo_config.run_name):
    mlflow.log_params({"sft_model": SFT_MODEL_NAME, "out_of_scope_pairs": len(train_out_of_scope_pairs), "shopping_pairs": len(train_shopping_pairs),
                       "val_pairs": len(val_pairs), "lora_r": lora_config.r, "lora_alpha": lora_config.lora_alpha})

    # TODO (4): 학습기로 끝까지 학습하세요
    ...

In [ ]:
# [자가채점] LoRA 설정, DPO 학습 설정, 학습기의 어댑터와 참조 모델, 끝까지 돈 갱신 횟수를 확인합니다
import math

from peft import PeftModel

# LoRA 설정 값
assert isinstance(lora_config, LoraConfig), "lora_config는 peft의 LoraConfig입니다. `...`를 채웠는지 확인하세요."
assert lora_config.r == 8 and lora_config.lora_alpha == 16 and lora_config.lora_dropout == 0.05, "r=8, lora_alpha=16, lora_dropout=0.05을 넣으세요."
assert lora_config.task_type == "CAUSAL_LM", "task_type=\"CAUSAL_LM\"을 넣으세요."

# DPO 학습 설정 값
assert isinstance(dpo_config, DPOConfig), "dpo_config는 trl의 DPOConfig입니다. `...`를 채웠는지 확인하세요."
assert abs(dpo_config.beta - 0.1) < 1e-12, f"beta가 {dpo_config.beta}입니다. 0.1을 넣으세요."
assert dpo_config.loss_type == ["sigmoid", "sft"], f"loss_type이 {dpo_config.loss_type}입니다. [\"sigmoid\", \"sft\"]를 넣으세요(DPOConfig 기본값은 [\"sigmoid\"]입니다)."
assert abs(dpo_config.learning_rate - 5e-5) < 1e-12, f"learning_rate가 {dpo_config.learning_rate}입니다. 5e-5를 넣으세요(DPOConfig 기본값은 1e-6입니다)."
assert dpo_config.num_train_epochs == 3 and dpo_config.max_steps == -1, "num_train_epochs=3를 넣고 max_steps는 넣지 마세요."
assert dpo_config.per_device_train_batch_size == 4 and dpo_config.per_device_eval_batch_size == 4, "per_device_train_batch_size=4, per_device_eval_batch_size=4를 넣으세요."
assert dpo_config.max_length == 512 and dpo_config.seed == 0 and dpo_config.gradient_checkpointing is False, "max_length=512, seed=0, gradient_checkpointing=False를 넣으세요."
assert dpo_config.eval_strategy == "epoch" and dpo_config.save_strategy == "epoch" and dpo_config.save_only_model is True, "eval_strategy=\"epoch\", save_strategy=\"epoch\", save_only_model=True를 넣으세요(에폭마다 어댑터를 저장합니다)."
assert dpo_config.logging_steps == 5 and dpo_config.eval_on_start is True, "logging_steps=5, eval_on_start=True를 넣으세요."
assert "mlflow" in dpo_config.report_to and dpo_config.run_name == "commerce_dpo_beta_0.1", "report_to=[\"mlflow\"], run_name=\"commerce_dpo_beta_0.1\"를 넣으세요."
assert dpo_config.output_dir.endswith("commerce_dpo_beta0.1"), f"output_dir가 {dpo_config.output_dir}입니다. str(output_dir / \"commerce_dpo_beta0.1\")를 넣으세요."

# 학습기: 어댑터를 붙인 모델, 참조 모델 없음, 학습, 검증 데이터
assert isinstance(trainer, DPOTrainer) and trainer.args is dpo_config, "trainer는 dpo_config를 args로 넘긴 DPOTrainer입니다."
assert isinstance(trainer.model, PeftModel), "trainer.model에 어댑터가 없습니다. DPOTrainer에 peft_config=lora_config를 넘기세요."
assert trainer.model.peft_config["default"].r == 8 and trainer.model.peft_config["default"].lora_alpha == 16, "trainer.model의 어댑터 r, lora_alpha가 이 문항 (1)과 다릅니다. (1)의 lora_config를 peft_config로 넘기세요."

# PEFT 는 "all-linear" 를 어댑터를 붙인 층 이름 집합으로 바꿔 저장합니다. 어텐션 층만 고르면 MLP 층(gate_proj, down_proj)이 빠집니다
adapter_targets = " ".join(sorted(trainer.model.peft_config["default"].target_modules))
assert "q_proj" in adapter_targets and "gate_proj" in adapter_targets and "down_proj" in adapter_targets, f"어댑터를 붙인 층이 {adapter_targets}입니다. (1)에 target_modules=\"all-linear\"를 넣어 lm_head 를 뺀 모든 선형 층에 붙이세요."

assert trainer.ref_model is None, "trainer.ref_model이 None이 아닙니다. ref_model을 넘기지 마세요(어댑터를 끈 같은 모델이 참조 모델입니다)."
assert len(trainer.train_dataset) == len(train_pairs) and trainer.eval_dataset is not None and len(trainer.eval_dataset) == len(val_pairs), "학습기의 학습, 검증 데이터 행 수가 train_pairs, val_pairs와 다릅니다. train_dataset=train_pairs, eval_dataset=val_pairs를 넘기세요."

# 끝까지 돈 갱신 횟수
expected_steps = math.ceil(len(train_pairs) / 4) * 3
assert trainer.state.global_step == expected_steps, f"trainer.state.global_step이 {trainer.state.global_step}입니다(기대값 {expected_steps}). (4)에서 trainer.train()이 끝까지 돌았는지 확인하세요."
print("✅ 통과!")

## 4. 학습 기록과 정렬 후 답: log_history, eval

**목적:** 학습이 선호를 배웠는지(검증 보상 지표)와 실제 답이 거절로 바뀌었는지를 함께 확인합니다.

**요구사항**
- (1) `trainer.state.log_history`를 `pd.DataFrame`으로 바꾼 **`log_table`** 을 만듭니다.
- (2) `log_table`에서 `"eval_loss"`가 비어 있지 않은 행만 남기고 `"epoch"`, `"eval_loss"`, `"eval_rewards/chosen"`, `"eval_rewards/rejected"`, `"eval_rewards/margins"`, `"eval_rewards/accuracies"` 여섯 열을 이 순서로 고른 **`reward_table`** 을 만듭니다(학습 전과 에폭 1~3, 4행).
- (3) 생성 전에 `trainer.model`을 평가 모드로 바꿉니다(`trainer.model.training`이 `False`).
- (4) `answer_check_questions(trainer.model)`의 답을 `check_table`의 새 열 `"정렬 모델 답변"`에 담습니다. 열은 `"종류"`, `"질문"`, `"정답 답변"`, `"SFT 모델 답변"`, `"정렬 모델 답변"` 순서이고, 답마다 비어 있지 않고 질문 글과 다른 문자열이며, 다섯 답 전체가 `"SFT 모델 답변"`과도, `reference_answers`와도 같지 않아야 합니다.
- (5) `"종류"`가 `"범위 밖 질문(검증)"`인 두 행의 `"정렬 모델 답변"` 가운데 `"답할 수 없"`가 든 답의 수를 정수(`int`) **`dpo_refusal_count`** 에 담습니다.

<details><summary>힌트</summary>

```text
접근방법: 교안 01 5-4 학습 기록 읽기, 5-5 정렬 전후 답 셀과 같은 방법입니다.
세부구현:
1. eval_loss 가 있는 행만 남길 때는 그 열이 빈 행을 빼는 DataFrame 메서드를 쓰고, 열 이름 리스트로 여섯 열을 고릅니다.
2. 생성 전에 학습기의 모델을 평가 모드로 바꾸고, 준비의 answer_check_questions 로 그 모델에 묻습니다.
3. 거절 수는 1번 (2)와 같은 방법으로 "정렬 모델 답변" 열에서 셉니다.
```

</details>

In [ ]:
# TODO (1): 학습 기록을 테이블 log_table 로 바꾸세요
log_table = ...

# TODO (2): 검증 행만 남기고 지문의 여섯 열을 골라 reward_table 을 만드세요
reward_table = ...
display(reward_table.round(4))

In [ ]:
# TODO (3): 생성 전에 trainer.model 을 평가 모드로 바꾸세요
...

# TODO (4): 같은 질문 다섯 개의 정렬 후 답을 check_table 의 새 열 "정렬 모델 답변" 에 담으세요
check_table["정렬 모델 답변"] = ...
display(check_table)

In [ ]:
# TODO (5): 범위 밖 질문 두 행의 "정렬 모델 답변" 가운데 거절 키워드가 든 답의 수를 dpo_refusal_count 에 담으세요
dpo_refusal_count = ...
print("범위 밖 질문 2개 가운데 거절한 답: SFT 모델", sft_refusal_count, "| 정렬 모델", dpo_refusal_count)

In [ ]:
# [자가채점] 검증 보상 지표 테이블, 평가 모드, 정렬 후 답 열과 거절 수를 확인합니다
# 학습 기록과 검증 보상 지표 테이블
assert isinstance(log_table, pd.DataFrame), "log_table은 trainer.state.log_history를 pd.DataFrame으로 바꾼 테이블입니다. `...`를 채웠는지 확인하세요."
assert len(log_table) == len(trainer.state.log_history), "log_table은 trainer.state.log_history의 기록마다 한 행입니다."
assert isinstance(reward_table, pd.DataFrame), "reward_table은 log_table에서 고른 테이블입니다. `...`를 채웠는지 확인하세요."
assert list(reward_table.columns) == ["epoch", "eval_loss", "eval_rewards/chosen", "eval_rewards/rejected", "eval_rewards/margins", "eval_rewards/accuracies"], f"reward_table의 열이 {list(reward_table.columns)}입니다. 지문의 여섯 열만 이 순서로 남기세요."
assert [round(epoch) for epoch in reward_table["epoch"]] == [0, 1, 2, 3], f"reward_table의 epoch이 {list(reward_table['epoch'])}입니다. eval_loss가 있는 검증 행만 남기고, eval_on_start=True, eval_strategy=\"epoch\"를 확인하세요."
for column in ["eval_rewards/chosen", "eval_rewards/rejected", "eval_rewards/margins"]:
    assert abs(reward_table[column].iloc[0]) < 0.01, f"학습 전 {column}이 0에서 멉니다. 학습 전에는 어댑터를 끈 모델과 학습 중 모델이 같아야 합니다. 학습을 다시 했다면 제공 코드의 SFT 모델 셀부터 다시 실행해 어댑터가 없는 SFT 모델로 학습하고, 3번 학습기에 ref_model을 넘기지 않았는지 확인하세요."

# 평가 모드
assert trainer.model.training is False, "trainer.model이 학습 모드입니다. 생성 전에 trainer.model을 평가 모드로 바꾸세요."

# 정렬 모델 답변 열: 비어 있지 않고 질문 글과 다르며, 다섯 답 전체가 SFT 모델 답변을 옮긴 것이 아닌지
assert list(check_table.columns) == ["종류", "질문", "정답 답변", "SFT 모델 답변", "정렬 모델 답변"], f"check_table의 열이 {list(check_table.columns)}입니다. 1번 테이블에 \"정렬 모델 답변\" 열 하나를 더하세요."
for question, answer in zip(check_questions, check_table["정렬 모델 답변"]):
    assert isinstance(answer, str) and answer.strip() != "", "\"정렬 모델 답변\"이 비어 있습니다. answer_check_questions(trainer.model)의 반환값을 담으세요."
    assert answer != question, "\"정렬 모델 답변\"에 질문 글이 들어 있습니다. answer_check_questions(trainer.model)의 반환값을 담으세요."
assert list(check_table["정렬 모델 답변"]) != list(check_table["SFT 모델 답변"]), "\"정렬 모델 답변\"이 \"SFT 모델 답변\"과 다섯 개 모두 같습니다. 학습한 trainer.model 에 다시 물었는지 확인하세요."
assert list(check_table["정렬 모델 답변"]) != reference_answers, "\"정렬 모델 답변\"이 \"정답 답변\"과 다섯 개 모두 같습니다. reference_answers가 아니라 answer_check_questions(trainer.model)의 반환값을 담으세요."

# 범위 밖 두 행의 거절 수: 두 행만 세므로 0부터 2 사이의 정수
assert isinstance(dpo_refusal_count, int), f"dpo_refusal_count가 {dpo_refusal_count!r}입니다. `...`를 채우고 거절한 답의 수를 정수로 담으세요(`.sum()`으로 셌다면 `int()`로 바꾸세요)."
assert 0 <= dpo_refusal_count <= 2, f"dpo_refusal_count가 {dpo_refusal_count}입니다. \"종류\"가 \"범위 밖 질문(검증)\"인 두 행의 \"정렬 모델 답변\"만 세므로 0부터 2 사이입니다."
print("✅ 통과!")